# vamprhyme-llm: LoRA fine-tuning on Colab

Runtime > Change runtime type > **T4 GPU**. See `finetune.md` for what each cell does.

## 1. Check the GPU and install libraries

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q -U transformers peft
!pip uninstall -y -q torchao
import torch, transformers, peft
print("PyTorch", torch.__version__, "| transformers", transformers.__version__, "| peft", peft.__version__)
print("CUDA available:", torch.cuda.is_available())

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

RUNS = "/content/drive/MyDrive/vamprhyme/runs"
!mkdir -p {RUNS}

## 3. Get the project

In [ ]:
import os
from google.colab import userdata

REPO = "YOUR_GITHUB_USERNAME/vamprhyme-llm"
PRIVATE = True

if PRIVATE:
    url = f"https://{userdata.get('GITHUB_TOKEN')}@github.com/{REPO}.git"
else:
    url = f"https://github.com/{REPO}.git"

if os.path.isdir("/content/vamprhyme-llm"):
    !git -C /content/vamprhyme-llm pull
else:
    !git clone -q {url} /content/vamprhyme-llm

%cd /content/vamprhyme-llm

## 4. Copy the fine-tuning data from Drive

In [ ]:
!unzip -oq /content/drive/MyDrive/vamprhyme/vamprhyme-sft.zip -d /content/vamprhyme-llm
!wc -l data/sft/train.jsonl data/sft/val.jsonl

## 5. Baseline: the pretrained model before fine-tuning

In [ ]:
!python chat.py --max-new-tokens 250 \
    "Write a nursery rhyme." \
    "Write a passage of Victorian gothic horror." \
    "Write a Dracula version of Twinkle, Twinkle, Little Star."

## 6. Fine-tune with LoRA

In [ ]:
%env PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True
!python finetune_lora.py --out {RUNS}/lora-styles

## 7. Compare base and fine-tuned

In [ ]:
!python chat.py --adapter {RUNS}/lora-styles/adapter --compare --max-new-tokens 250 \
    "Write a nursery rhyme." \
    "Write a passage of Victorian gothic horror." \
    "Write a Dracula version of Twinkle, Twinkle, Little Star."

## 8. Loss curves

In [ ]:
from IPython.display import Image
Image(f"{RUNS}/lora-styles/loss.png")